# ЛР2 — запуск целиком
Включите GPU T4. Выполните первую ячейку до `СЕРВЕР ГОТОВ`, затем вторую.

In [ ]:
import os, pathlib, shutil, subprocess, sys, time, urllib.request, json
repo = 'https://github.com/marchukmarina-creator/lr2-rag-scifact.git'
project = pathlib.Path('/content/lr2-rag-scifact')
if not project.exists(): subprocess.run(['git', 'clone', repo, str(project)], check=True)
else: subprocess.run(['git', '-C', str(project), 'pull', '--ff-only'], check=True)
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.33.0', '-r', 'requirements.txt'], check=True)
def ready():
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=3) as r: return any(x.get('id') == 'Qwen/Qwen3-4B-Instruct-2507' for x in json.load(r).get('data', []))
    except Exception: return False
if not ready():
    log = open('/content/vllm_lr2.log', 'w', encoding='utf-8')
    command = [shutil.which('vllm') or 'vllm', 'serve', 'Qwen/Qwen3-4B-Instruct-2507', '--revision', 'cdbee75f17c01a7cc42f958dc650907174af0554', '--dtype', 'half', '--max-model-len', '32768', '--gpu-memory-utilization', '0.90', '--max-num-seqs', '32', '--max-num-batched-tokens', '2048', '--enforce-eager', '--enable-prefix-caching', '--host', '127.0.0.1', '--port', '8000']
    env = os.environ.copy(); env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
    server = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(240):
        if ready(): break
        if server.poll() is not None:
            log.flush(); raise RuntimeError(pathlib.Path('/content/vllm_lr2.log').read_text(encoding='utf-8')[-12000:])
        time.sleep(5)
if not ready(): raise RuntimeError('Сервер не запустился. Покажите последние строки /content/vllm_lr2.log')
print('СЕРВЕР ГОТОВ')

In [ ]:
import os, pathlib, subprocess, sys
from google.colab import files
project = pathlib.Path('/content/lr2-rag-scifact'); os.chdir(project)
steps = [[sys.executable, 'lr2.py', x] for x in ['prepare', 'check', 'run', 'analyze', 'pack']]
for step in steps:
    print('\n>>>', ' '.join(step[1:])); subprocess.run(step, check=True)
archive = project / 'lr2_lr2_final_artifacts.zip'
files.download(str(archive))
print('ЛР2 ВЫПОЛНЕНА. Архив с результатами скачивается.')